# Notebook 02 — Data Exploration

**Goal:** Let the data justify every H2 index decision.
We are NOT exploring passage retrieval here — H1 is dropped.
We focus entirely on understanding what the 38 training queries
and 263k documents need from an enriched index.

**What we answer:**
1. Query type distribution — what kinds of queries do we have?
2. Vocabulary gap — do query terms appear in relevant documents?
3. Field coverage — which fields are reliable enough to index?
4. Relevant vs irrelevant — which fields discriminate best?
5. Abstract length — how long are documents? (one cell, one number)
6. Summary — data-driven justification for H2 index design

---

In [ ]:
import json, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from elasticsearch import Elasticsearch

es = Elasticsearch(host='elasticsearch')
print('ES:', es.info()['version']['number'])

CORPUS        = '/work/data/FIR-s05-medline.json'
TRAIN_QUERIES = '/work/data/FIR-s05-training-queries-simple.txt'
TEST_QUERIES  = '/work/data/test-queries-simple.txt'
QRELS         = '/work/data/FIR-s05-training-qrels.txt'
INDEX         = 'medline'

def load_queries(path):
    queries = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1].strip('"')
    return queries

def load_qrels(path):
    qrels = defaultdict(dict)
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

train_queries = load_queries(TRAIN_QUERIES)
test_queries  = load_queries(TEST_QUERIES)
qrels         = load_qrels(QRELS)

print(f'Training queries: {len(train_queries)}')
print(f'Test queries:     {len(test_queries)}')
print(f'Qrels topics:     {len(qrels)}')
print(f'Note: evaluation runs on training queries only (qrels available)')

---
## 1. Query type distribution — both splits

In [ ]:
SPECIES_MAP = {
    'Homo sapiens':            ['homo sapiens', 'human'],
    'Mus musculus':            ['mus musculus', 'mouse'],
    'Rattus norvegicus':       ['rattus norvegicus', 'rat'],
    'Drosophila melanogaster': ['drosophila'],
}

AMBIGUOUS = {'fat', 'numb', 'reaper', 'brahma', 'groucho'}

JUNK_WORDS = {
    'alpha','beta','gamma','delta','epsilon',
    'ligand','mouse','human','rat','drosophila','cerevisiae',
    'protein','factor','receptor','inhibitor','activator',
    'homolog','isoform','type','subunit','domain',
    'family','member','related','associated','like','avian',
}

DESCRIPTION_PHRASES = [
    'inactivator','tyrosine kinase','binding protein',
    'growth factor','growth inhibitory','neurotrophic',
    'small gtp','coagulation','stiff man','hyperekplexia',
    'startle disease','rho family','oncogene ligand',
    'colony-stimulating','cutaneous and mucosal',
    'venous malformations','sphingolipid',
]

def is_real_alias(text):
    text = text.strip()
    if not text or len(text) <= 1:      return False
    if len(text.split()) > 3:           return False
    if text.lower() in JUNK_WORDS:      return False
    if any(p in text.lower() for p in DESCRIPTION_PHRASES): return False
    if text[-1] in (',','(','/'): return False
    return True

def classify_query(qid, qtext):
    text_lower = qtext.lower()
    species = 'unknown'
    for sp, pats in SPECIES_MAP.items():
        if any(p in text_lower for p in pats):
            species = sp; break

    gene_part = re.sub(
        r'\s+in\s+(' + '|'.join(SPECIES_MAP.keys()) + r')',
        '', qtext, flags=re.IGNORECASE
    ).strip().strip('"')

    alias_list = []
    for a in re.findall(r'\(([^)]+)\)', gene_part):
        for part in a.split(','):
            if is_real_alias(part.strip()):
                alias_list.append(part.strip())

    gene_base  = re.sub(r'\([^)]*\)', '', gene_part).strip()
    word_count = len(gene_base.split())
    is_ambig   = gene_base.lower().strip() in AMBIGUOUS

    if is_ambig:     qtype = 'gene_symbol_ambiguous'
    elif alias_list: qtype = 'gene_with_aliases'
    elif word_count <= 3 and not re.search(r'[a-z]{5,}', gene_base):
                     qtype = 'gene_symbol_simple'
    else:            qtype = 'gene_descriptor'

    return {
        'qid': qid, 'raw': qtext, 'gene_part': gene_part,
        'gene_base': gene_base, 'species': species,
        'aliases': alias_list, 'qtype': qtype,
        'is_ambig': is_ambig, 'word_count': word_count,
    }

# classify both splits
train_cls = {qid: classify_query(qid, qt) for qid, qt in train_queries.items()}
test_cls  = {qid: classify_query(qid, qt) for qid, qt in test_queries.items()}
all_cls   = {**train_cls, **test_cls}

# distribution comparison
train_dist = Counter(c['qtype'] for c in train_cls.values())
test_dist  = Counter(c['qtype'] for c in test_cls.values())

print(f'{"Type":<26} {"Train":>8} {"Test":>8} {"Total":>8}')
print('─' * 55)
all_types = sorted(set(list(train_dist) + list(test_dist)))
for qtype in all_types:
    t  = train_dist.get(qtype, 0)
    ts = test_dist.get(qtype, 0)
    print(f'{qtype:<26} {t:>8} {ts:>8} {t+ts:>8}')
print(f'{"TOTAL":<26} {len(train_cls):>8} {len(test_cls):>8} {len(all_cls):>8}')

# species distribution
print(f'\n{"Species":<28} {"Train":>8} {"Test":>8}')
print('─' * 46)
train_sp = Counter(c['species'] for c in train_cls.values())
test_sp  = Counter(c['species'] for c in test_cls.values())
for sp in sorted(set(list(train_sp)+list(test_sp))):
    print(f'{sp:<28} {train_sp.get(sp,0):>8} {test_sp.get(sp,0):>8}')

In [ ]:
# visualise distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# query type
qtypes = all_types
x = np.arange(len(qtypes))
axes[0].bar(x - 0.2, [train_dist.get(q,0) for q in qtypes],
            0.4, label='Train', color='#2E75B6')
axes[0].bar(x + 0.2, [test_dist.get(q,0) for q in qtypes],
            0.4, label='Test',  color='#1ABC9C')
axes[0].set_xticks(x)
axes[0].set_xticklabels(qtypes, rotation=15, ha='right')
axes[0].set_ylabel('Count')
axes[0].set_title('Query Type Distribution')
axes[0].legend()

# species
species_list = sorted(set(list(train_sp)+list(test_sp)))
x2 = np.arange(len(species_list))
axes[1].bar(x2 - 0.2, [train_sp.get(s,0) for s in species_list],
            0.4, label='Train', color='#2E75B6')
axes[1].bar(x2 + 0.2, [test_sp.get(s,0) for s in species_list],
            0.4, label='Test',  color='#1ABC9C')
axes[1].set_xticks(x2)
axes[1].set_xticklabels([s.split()[0] for s in species_list], rotation=15, ha='right')
axes[1].set_ylabel('Count')
axes[1].set_title('Species Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('/work/runs/query_distribution.png', dpi=120)
plt.show()

# key finding
desc_pct = (train_dist.get('gene_descriptor',0) / len(train_cls)) * 100
alias_count = sum(1 for c in train_cls.values() if c['aliases'])
print(f'\n── Key finding ──────────────────────────────────────────────')
print(f'{desc_pct:.0f}% of training queries are gene_descriptor')
print(f'Only {alias_count} training queries have real gene aliases')
print(f'→ H2 recall gain will come primarily from biomedical analyzer')
print(f'  quality (stemming, MeSH matching) not alias expansion')
print(f'→ Per-type breakdown is more informative than aggregate MAP')

---
## 2. Vocabulary gap — does query text appear in relevant documents?

In [ ]:
# For each training query: check whether gene name tokens
# appear in the relevant documents
# If NOT → vocabulary gap → index enrichment needed

def get_doc_text(es, index, doc_id):
    try:
        r = es.get(index=index, id=doc_id, _source=['TI','AB'])
        return f"{r['_source'].get('TI','')} {r['_source'].get('AB','')}".lower()
    except Exception:
        return ''

print('Measuring vocabulary gap on training queries...')
print('(makes ES calls — takes ~2 min)\n')

gap_results = []
for qid, c in train_cls.items():
    rel_docs = [d for d, r in qrels.get(qid, {}).items() if r > 0][:3]
    if not rel_docs:
        continue

    # gene tokens from query (no species)
    gene_tokens = set(
        re.findall(r'[a-z0-9]+', c['gene_base'].lower())
    ) - {'in','the','of','and','or','a','an'}

    gap_docs = 0
    for doc_id in rel_docs:
        doc_text = get_doc_text(es, INDEX, doc_id)
        if not doc_text:
            continue
        missing = [t for t in gene_tokens
                   if t not in doc_text and len(t) > 3]
        if missing:
            gap_docs += 1

    gap_pct = gap_docs / len(rel_docs) * 100 if rel_docs else 0
    gap_results.append({
        'qid':     qid,
        'qtype':   c['qtype'],
        'species': c['species'].split()[0],
        'gene':    c['gene_base'][:40],
        'gap_pct': gap_pct,
        'has_gap': gap_pct > 0,
    })

df_gap = pd.DataFrame(gap_results)

print(f'{"QID":<6} {"Gap%":>6}  {"Type":<22} {"Gene"}')
print('─' * 70)
for _, r in df_gap.sort_values('gap_pct', ascending=False).iterrows():
    marker = '⚠' if r['gap_pct'] > 50 else ('~' if r['gap_pct'] > 0 else ' ')
    print(f"{r['qid']:<6} {r['gap_pct']:>5.0f}%  {marker} {r['qtype']:<22} {r['gene']}")

gap_count = df_gap['has_gap'].sum()
print(f'\nQueries with vocabulary gap: {gap_count}/{len(df_gap)} ({gap_count/len(df_gap)*100:.0f}%)')

In [ ]:
# gap by query type
fig, ax = plt.subplots(figsize=(8, 4))
type_gap = df_gap.groupby('qtype')['gap_pct'].agg(['mean','count'])
colors = ['#E74C3C' if g > 30 else '#F39C12' if g > 10 else '#27AE60'
          for g in type_gap['mean']]
bars = ax.bar(type_gap.index, type_gap['mean'], color=colors, edgecolor='white')
for bar, (idx, row) in zip(bars, type_gap.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f"{row['mean']:.0f}% (n={int(row['count'])})",
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Mean vocabulary gap %')
ax.set_title('Vocabulary Gap by Query Type\n(% of relevant docs missing query tokens)')
ax.set_xlabel('Query type')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('/work/runs/vocab_gap.png', dpi=120)
plt.show()

print('Red   = high gap → enrichment critical for this type')
print('Orange = moderate gap')
print('Green  = low gap → query terms already in relevant docs')

---
## 3. Field coverage — which fields are reliable?

In [ ]:
# Sample 2000 docs to measure field completeness
field_counts = Counter()
ab_lengths   = []
mesh_counts  = []
sampled      = 0
SAMPLE_SIZE  = 2000

with open(CORPUS) as f:
    while sampled < SAMPLE_SIZE:
        meta = f.readline()
        doc  = f.readline()
        if not meta or not doc:
            break
        d = json.loads(doc)
        for field in d:
            if d[field]:
                field_counts[field] += 1
        if d.get('AB'):
            ab_lengths.append(len(d['AB'].split()))
        mesh_counts.append(len(d.get('MH', [])))
        sampled += 1

print(f'Sampled {sampled:,} documents\n')
print(f'{"Field":<8} {"Coverage":>10} {"Use in index"}')
print('─' * 55)

field_decisions = {
    'TI':   'text field  → biomedical_prose analyzer  boost 2x',
    'AB':   'text field  → biomedical_prose analyzer  boost 1x',
    'MH':   'concept_context → prose analyzer         boost 1.5x',
    'RN':   'substance_names → extracted from parens  boost 1x',
    'TA':   'keyword     → journal tier filter (future)',
    'DP':   'integer     → pub_year for function_score',
    'PMID': 'keyword     → document ID for qrels matching',
    'FAU':  'skip        → author names not useful for retrieval',
    'AD':   'skip        → affiliation not useful',
    'LA':   'skip        → language (filter to EN at query time)',
}

for field, count in sorted(field_counts.items(), key=lambda x: -x[1]):
    pct      = count / sampled * 100
    decision = field_decisions.get(field, 'not indexed')
    bar      = '█' * int(pct/5)
    print(f'{field:<8} {pct:>9.1f}%  {decision}')

print(f'\nAbstract length: mean={np.mean(ab_lengths):.0f} words, '
      f'median={np.median(ab_lengths):.0f}')
print(f'MeSH per doc:    mean={np.mean(mesh_counts):.1f}, '
      f'docs with MeSH={sum(1 for m in mesh_counts if m>0)/sampled*100:.0f}%')
print(f'\n→ H1 dropped: mean abstract {np.mean(ab_lengths):.0f} words is too short')
print(f'  for passage retrieval to add meaningful granularity')

---
## 4. Relevant vs irrelevant — which fields discriminate best?

In [ ]:
# This tells you the right field boosts for H2
# Fields that overlap more with query tokens in relevant docs
# should get higher boosts

print('Comparing relevant vs non-relevant docs...')
results = []

for qid in list(train_cls.keys())[:15]:  # first 15 queries
    qtext   = train_queries[qid]
    q_tokens = set(re.findall(r'[a-z0-9]+', qtext.lower())) \
               - {'in','the','of','and','or','a','an','homo',
                  'sapiens','mus','musculus','rattus','norvegicus','drosophila'}

    # relevant docs
    rel_ids = [d for d, r in qrels.get(qid, {}).items() if r > 0][:3]
    for doc_id in rel_ids:
        try:
            r = es.get(index=INDEX, id=doc_id, _source=['TI','AB','MH'])
            src = r['_source']
            ti_tok = set(re.findall(r'[a-z0-9]+', src.get('TI','').lower()))
            ab_tok = set(re.findall(r'[a-z0-9]+', src.get('AB','').lower()))
            mh_tok = set(re.findall(r'[a-z0-9]+',
                         ' '.join(src.get('MH',[])).lower()))
            results.append({
                'relevant':   True,
                'ti_overlap': len(q_tokens & ti_tok),
                'ab_overlap': len(q_tokens & ab_tok),
                'mh_overlap': len(q_tokens & mh_tok),
            })
        except Exception:
            pass

    # non-relevant (top retrieved not in qrels)
    resp = es.search(index=INDEX, body={
        'query': {'multi_match': {'query': qtext, 'fields': ['TI','AB']}},
        'size': 10
    })
    nonrel = [h for h in resp['hits']['hits']
              if h['_id'] not in set(rel_ids)][:3]
    for hit in nonrel:
        src = hit['_source']
        ti_tok = set(re.findall(r'[a-z0-9]+', src.get('TI','').lower()))
        ab_tok = set(re.findall(r'[a-z0-9]+', src.get('AB','').lower()))
        mh_tok = set(re.findall(r'[a-z0-9]+',
                     ' '.join(src.get('MH',[])).lower()))
        results.append({
            'relevant':   False,
            'ti_overlap': len(q_tokens & ti_tok),
            'ab_overlap': len(q_tokens & ab_tok),
            'mh_overlap': len(q_tokens & mh_tok),
        })

df = pd.DataFrame(results)
print('\nMean query-token overlap by field:')
summary = df.groupby('relevant')[['ti_overlap','ab_overlap','mh_overlap']].mean().round(2)
summary.index = ['Non-relevant','Relevant']
print(summary)

print('\nRelevant/Non-relevant ratio (higher = better discriminator):')
for col, label in [('ti_overlap','TI'),('ab_overlap','AB'),('mh_overlap','MH')]:
    rel    = summary.loc['Relevant', col]
    nonrel = summary.loc['Non-relevant', col]
    ratio  = rel / (nonrel + 0.001)
    boost  = min(round(ratio, 1), 3.0)
    print(f'  {label}: ratio={ratio:.2f}x → suggested boost {boost}x')

In [ ]:
# visualise field discrimination
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (col, label) in zip(axes, [
    ('ti_overlap', 'TI (Title)'),
    ('ab_overlap', 'AB (Abstract)'),
    ('mh_overlap', 'MH (MeSH)'),
]):
    rel    = df[df['relevant'] == True][col]
    nonrel = df[df['relevant'] == False][col]
    ax.hist(rel,    bins=10, alpha=0.6, label='Relevant',     color='#27AE60')
    ax.hist(nonrel, bins=10, alpha=0.6, label='Non-relevant', color='#E74C3C')
    ax.set_title(label)
    ax.set_xlabel('Token overlap with query')
    ax.legend(fontsize=8)

plt.suptitle('Field Discrimination: Relevant vs Non-relevant')
plt.tight_layout()
plt.savefig('/work/runs/field_discrimination.png', dpi=120)
plt.show()

print('Fields where relevant docs overlap more → higher boost in H2 index')

---
## 5. Relevant doc count per query — drives metric choice

In [ ]:
rel_counts = {qid: sum(1 for r in docs.values() if r > 0)
              for qid, docs in qrels.items()}
counts = list(rel_counts.values())

print(f'Relevant docs per query:')
print(f'  Mean:   {np.mean(counts):.1f}')
print(f'  Median: {np.median(counts):.0f}')
print(f'  Min:    {np.min(counts)}')
print(f'  Max:    {np.max(counts)}')
print(f'  Std:    {np.std(counts):.1f}')

variance = np.std(counts) / np.mean(counts)
print(f'\nCoefficient of variation: {variance:.2f}')
if variance > 0.5:
    print('HIGH variance → P@10 is unfair across topics')
    print('Use R-Precision as a secondary metric alongside P@5')
else:
    print('Low variance → P@10 is reasonably fair')

---
## 6. Summary — H2 index design justified by data

In [ ]:
desc_pct    = train_dist.get('gene_descriptor',0) / len(train_cls) * 100
alias_count = sum(1 for c in train_cls.values() if c['aliases'])
ambig_count = train_dist.get('gene_symbol_ambiguous', 0)
gap_pct     = df_gap['has_gap'].sum() / len(df_gap) * 100
mean_ab     = np.mean(ab_lengths)
mean_mesh   = np.mean(mesh_counts)

print('='*65)
print('H2 INDEX DESIGN — DATA-DRIVEN JUSTIFICATION')
print('='*65)
print(f'''
Finding                          Decision for H2 index
─────────────────────────────────────────────────────────────────
{desc_pct:.0f}% queries are gene_descriptor   → biomedical_prose analyzer
  (long NCBI full names)           minimal_english stemmer
                                   captures morphological variants

Only {alias_count}/38 queries have         → Entrez Gene expansion helps
  real gene aliases                  these {alias_count} queries specifically
                                   per-type breakdown needed

{ambig_count} ambiguous Drosophila queries  → hard species filter for these
  (fat, numb, brahma, reaper)       soft boost for all other types

{gap_pct:.0f}% queries have vocab gap      → enriched fields close this gap:
                                   gene_context (exact identifiers)
                                   concept_context (MeSH + substances)

MeSH in {mean_mesh:.0f} headings/doc (avg)    → concept_context field
  expert-curated annotation          biomedical_prose analyzer
                                   boost 2.0x (trusted annotation)

Abstract mean = {mean_ab:.0f} words          → H1 DROPPED: too short for
                                   passage retrieval to add value
                                   Dai & Callan 2019 confirmed
─────────────────────────────────────────────────────────────────
SINGLE HYPOTHESIS: H2
Does a knowledge-enriched inverted index (gene_context +
concept_context + biomedical analyzer) improve recall@100
and MAP compared to plain BM25 on TREC Genomics 2005?

Expected: modest aggregate improvement (5-10% recall@100)
Largest gain: {alias_count} gene_with_aliases queries
Mechanism:    biomedical analyzer quality for descriptor queries
''')

# save classifications for use in notebooks 03 and 05
import json
with open('/work/data/query_classifications.json', 'w') as f:
    json.dump({qid: {k: v for k, v in c.items() if k != 'raw'}
               for qid, c in all_cls.items()}, f, indent=2)
print('Classifications saved → /work/data/query_classifications.json')